# BERT in Practice: Pre-trained Sentiment Models and the Fine-tuning Recipe

## 📚 Learning Objectives

By completing this notebook, you will:
- Use a pre-trained (already fine-tuned) DistilBERT model for sentiment classification
- Understand transfer learning with pre-trained models
- Walk through the Hugging Face fine-tuning recipe (Trainer/TrainingArguments code walkthrough — not executed here)
- Know what a fine-tuning run needs (labeled dataset, GPU time) before attempting one

## 🔗 Prerequisites

- ✅ `01_attention_transformers_bridge.ipynb` — what attention is, and BERT as a pretrained encoder
- ✅ `02_rnn_lstm_nlp.ipynb` — the sequence models transformers replaced
- ✅ Unit 3: Machine Learning for NLP (classification and evaluation concepts)

We use the model as a **pretrained black box** here; the deep-learning mechanics behind it arrive in **AIAT 122 (Course 08)**.

---

This notebook covers practical activities from **Course 07, Unit 4**:
- Fine-tuning BERT model for text classification using Hugging Face Transformers

---

## Introduction

**BERT (Bidirectional Encoder Representations from Transformers)** is a powerful pre-trained language model. Fine-tuning BERT allows us to adapt it for specific NLP tasks like text classification.

## 📥 Inputs & 📤 Outputs

**Inputs:** three short example sentences defined in the code, and the pretrained checkpoint `distilbert-base-uncased-finetuned-sst-2-english` from Hugging Face (~250MB download on first run; cached afterwards).

**Outputs:** a real sentiment label and confidence score for each sentence, computed by the pretrained model — plus a printed fine-tuning recipe, which is a code walkthrough and is **not executed** here.

---

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Try importing Hugging Face Transformers
try:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
    from transformers import pipeline
    HAS_TRANSFORMERS = True
    print("✅ Hugging Face Transformers available!")
except ImportError:
    HAS_TRANSFORMERS = False
    print("⚠️  Transformers not available. Install with: pip install transformers")

# (No TensorFlow needed here: the transformers pipeline runs on its PyTorch backend.)
print("\n✅ Libraries imported!")

✅ Hugging Face Transformers available!

✅ Libraries imported!


## Part 1: Using a Pre-trained BERT-family Model with Hugging Face


In [2]:
if HAS_TRANSFORMERS:
    print("=" * 60)
    print("Using a Pre-trained BERT-family Model")
    print("=" * 60)
    
    # DistilBERT is a smaller, faster BERT variant; this checkpoint is already
    # fine-tuned for sentiment analysis on the SST-2 dataset.
    model_name = "distilbert-base-uncased-finetuned-sst-2-english"
    
    print(f"\nLoading pretrained sentiment model: {model_name}")
    print("(a distilled BERT variant, already fine-tuned for sentiment)")
    print("Note: First run downloads the model (~250MB); later runs use the local cache")
    
    try:
        # Create a simple text classification pipeline
        classifier = pipeline("sentiment-analysis", model=model_name)
        
        # Test with sample text
        sample_texts = [
            "I love natural language processing!",
            "This course is challenging but interesting.",
            "Machine learning is difficult to understand."
        ]
        
        print("\n" + "-" * 60)
        print("Text Classification Results:")
        print("-" * 60)
        
        for text in sample_texts:
            result = classifier(text)[0]
            print(f"\nText: '{text}'")
            print(f"Label: {result['label']}, Score: {result['score']:.4f}")
        
        print("\n✅ Pre-trained BERT-family model (DistilBERT) used successfully!")
        
    except Exception as e:
        print(f"\nNote: Model download required. Error: {e}")
        print("To use BERT:")
        print("  1. Install: pip install transformers")
        print("  2. Model will download automatically on first use")
        print("  3. Requires internet connection for first run")
        
else:
    print("=" * 60)
    print("BERT with Hugging Face (Installation Required)")
    print("=" * 60)
    print("""
    To use BERT:
    
    1. Install Transformers:
       pip install transformers
    
    2. Load pre-trained model:
       from transformers import pipeline
       classifier = pipeline("sentiment-analysis")
       result = classifier("I love NLP!")
    
    3. Fine-tune for custom task:
       from transformers import AutoTokenizer, AutoModelForSequenceClassification
       tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
       model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased")
    """)


Using a Pre-trained BERT-family Model

Loading pretrained sentiment model: distilbert-base-uncased-finetuned-sst-2-english
(a distilled BERT variant, already fine-tuned for sentiment)
Note: First run downloads the model (~250MB); later runs use the local cache


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


------------------------------------------------------------
Text Classification Results:
------------------------------------------------------------

Text: 'I love natural language processing!'
Label: POSITIVE, Score: 0.9999

Text: 'This course is challenging but interesting.'
Label: POSITIVE, Score: 0.9970

Text: 'Machine learning is difficult to understand.'
Label: NEGATIVE, Score: 0.9992

✅ Pre-trained BERT-family model (DistilBERT) used successfully!


## Part 2: The Fine-tuning Recipe (code walkthrough — not executed)

The cell below prints the standard Hugging Face fine-tuning recipe so you can read it line by line.
Nothing is trained here: a real fine-tuning run needs a labeled dataset and GPU time (see the scope
note in the learning objectives).

In [3]:
if HAS_TRANSFORMERS:
    print("=" * 60)
    print("Fine-tuning BERT for Text Classification")
    print("=" * 60)
    
    print("\nFine-tuning Process:")
    print("  1. Load pre-trained BERT model")
    print("  2. Add classification head (num_labels)")
    print("  3. Prepare tokenized dataset")
    print("  4. Train on custom dataset")
    print("  5. Evaluate on test set")
    
    print("\n✅ Fine-tuning Steps:")
    print("""
    # Example code structure:
    
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    from transformers import Trainer, TrainingArguments
    
    # Load model and tokenizer
    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
    model = AutoModelForSequenceClassification.from_pretrained(
        "bert-base-uncased", num_labels=2  # binary classification
    )
    
    # Tokenize data
    train_encodings = tokenizer(train_texts, truncation=True, padding=True)
    
    # Training arguments
    training_args = TrainingArguments(
        output_dir='./results', num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        warmup_steps=500,
        logging_dir='./logs'
    )
    
    # Trainer
    trainer = Trainer(
        model=model, args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset
    )
    
    # Fine-tune
    trainer.train()
    """)
    
    print("\n📋 That is the full fine-tuning recipe — NOT executed here (it needs a labeled dataset and GPU time)")
else:
    print("Note: Install transformers to fine-tune BERT models")


Fine-tuning BERT for Text Classification

Fine-tuning Process:
  1. Load pre-trained BERT model
  2. Add classification head (num_labels)
  3. Prepare tokenized dataset
  4. Train on custom dataset
  5. Evaluate on test set

✅ Fine-tuning Steps:

    # Example code structure:

    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    from transformers import Trainer, TrainingArguments

    # Load model and tokenizer
    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
    model = AutoModelForSequenceClassification.from_pretrained(
        "bert-base-uncased", num_labels=2  # binary classification
    )

    # Tokenize data
    train_encodings = tokenizer(train_texts, truncation=True, padding=True)

    # Training arguments
    training_args = TrainingArguments(
        output_dir='./results', num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        warmup_steps=500,
        logging_dir='./logs'
   

## Summary

### Key Concepts:
1. **BERT**: Bidirectional Encoder Representations from Transformers
   - Pre-trained on large text corpora
   - Understands context bidirectionally
   - Can be fine-tuned for specific tasks

2. **Fine-tuning**: Adapt pre-trained BERT for downstream tasks
   - Text classification (sentiment, topic, etc.)
   - Named Entity Recognition (NER)
   - Question Answering
   - Text summarization

3. **Transfer Learning**: Use knowledge from pre-trained models
   - Faster training with fewer data
   - Better performance than training from scratch
   - Leverage large-scale pre-training

### Best Practices:
- Use appropriate pre-trained model (BERT, DistilBERT, RoBERTa)
- Fine-tune learning rate (typically 2e-5 to 5e-5)
- Use appropriate batch size (16-32)
- Monitor training to avoid overfitting
- Evaluate on validation set

**Reference:** Course 07, Unit 4: "Deep Learning for NLP" - BERT fine-tuning practical content
